# AI-vs-Human lexical baseline (TF-IDF + Naive Bayes) on PolitiFact++ & GossipCop++

Applies the **same technique** as `ai-generated-vs-human-text-95-accuracy.ipynb` to the two
LIFE benchmarks. **Task = provenance:** LLM-generated (MF + MR) = **AI (1)** vs human-written
(HF + HR) = **human (0)**.

- Pure lexical bag-of-words: strip `\n`/`'` → drop punctuation → remove stopwords →
  `CountVectorizer → TfidfTransformer → MultinomialNB`.
- **CPU only** — no GPU needed (Runtime → Change runtime type → CPU is fine). Independent of the
  LIFE fingerprint pipeline; reads the raw article text directly.
- **How to read it:** with a ~40/60 AI/human class balance, **macro-F1 and AI(1) recall** are the
  honest metrics — plain accuracy can look inflated by the majority (human) class.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

PROJECT_DIR    = '/content/drive/MyDrive/LIFE'
DATASET_ROOT   = f'{PROJECT_DIR}/dataset/data/Fakenews-dataset-main/Fakenews-dataset-main/Dataset'
POLITIFACT_DIR = f'{DATASET_ROOT}/PolitiFact++'
GOSSIPCOP_DIR  = f'{DATASET_ROOT}/GossipCop++'

os.chdir(PROJECT_DIR)  # so the relative script path below resolves
print('PolitiFact++ found:', os.path.isdir(POLITIFACT_DIR))
print('GossipCop++  found:', os.path.isdir(GOSSIPCOP_DIR))

## Run the baseline

Each cell prints the class balance, accuracy, and a per-class classification report.
sklearn / pandas / nltk are preinstalled on Colab; the script downloads the needed NLTK data on
first run. GossipCop++ (~20k articles) still finishes in well under a minute on CPU.

In [ ]:
!python ai_vs_human_code/run_life_baseline.py --data_dir "{POLITIFACT_DIR}" --name PolitiFact++

In [ ]:
!python ai_vs_human_code/run_life_baseline.py --data_dir "{GOSSIPCOP_DIR}" --name GossipCop++

## Notes
- **Labels are provenance, not veracity:** AI = MF/MR (GPT-3.5), human = HF/HR. To try other
  cuts (fake-vs-real, or LIFE's MF-vs-MR), edit `FILE_LABELS` in `run_life_baseline.py`.
- This baseline uses **no** LIFE machinery (no key sentences, no LLaMA perplexity fingerprints).
  It's a cheap lexical reference point: how much of the AI-vs-human signal already lives in the
  surface vocabulary. Where it falls short is where LIFE's fingerprint method earns its keep.
- Reproducibility: fixed `--seed 42`, `--test_size 0.3` (same as the Kaggle notebook).